In [22]:
from langgraph.graph import StateGraph
from typing import TypedDict
import os 
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from chat_memory import get_chat_history
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.messages import HumanMessage
from video_processing import analyze_video
from vectorstore_setup import clean_classification_text
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langgraph.graph import StateGraph, START, END
import tempfile

embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
vectorstore = Chroma(persist_directory="path/to/your/chroma_db", embedding_function=embeddings)

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGSMITH_API_KEY")



# when a query comes in, use this model to embed the query text so you can compare it against the vectors already stored.
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
vectorstore = Chroma(persist_directory="/Users/chandlershortlidge/Desktop/Ironhack/fitness-form-coach/chroma_db", 
                                    embedding_function=embeddings)


# Whisper speach to text 
from openai import OpenAI
client = OpenAI()




In [23]:
router_prompt = ChatPromptTemplate.from_messages([
    ("system", """Your job is to classify user queries into two buckets: 
     - memory
     - memory and vectorstore
     
     *Guidelines*
     Questions regarding exercises, exercise form, or technique: vectorstore & memory
     General question or statements: memory
   
     *Respond to queries with only the correct class*
     Examples: 
     User: how should I grip the bar for bench press?
     Agent: vectorstore & memory

     User: what muscles should I feel during the Romanian dead lift?
     Agent: vectorstore & memory

     User: How's this? 
     Agent: vectorstore & memory

     User: Is this good squat technique? 
     Agent: vectorstore & memory

     User: Was my bar path better?
     Agent: vectorstore & memory

     User: thakns for your help!
     Agent: memory

"""),

     ("human", "{query}")
     
])

llm = ChatOpenAI(model='gpt-4o')

output_parser = StrOutputParser()

router_chain = router_prompt | llm | output_parser


In [24]:
# Define the state

# the state is a dictionary

class GraphState(TypedDict):

    session_id: int 

    user_query: str

    user_video: str

    classification_image: str

    classified_keywords: str

    top_k_chunks: str

    encoded_images: list[str]

    response: str




In [25]:
workflow = StateGraph(GraphState) 

In [26]:
def video_encoder_node(state: GraphState):
    user_video = state["user_video"]
    user_video_encoded = analyze_video(filepath_in=user_video, frame_count=15, max_seconds=10) # encodes video
    encoded_images = [] # creates a list of encoded images for LLM
    for images in user_video_encoded:
        encoded_images.append({"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{images}"}})
        classification_image = user_video_encoded[0] 
    
    return {"classification_image": classification_image, "encoded_images": encoded_images}


In [27]:
# video classification router NODE

def video_classification_node(state: GraphState):
    classification_image = state["classification_image"]

    router_llm = ChatOpenAI(model='gpt-4o')

    response = router_llm.invoke([

    {"role": "user", "content": 

    [{"type": "text", "text":  """Your job is to analyze images of users working out for proper form, and list the key checkpoints of their to body evaluate. 
    Give me ONLY the bodypart checkpoints. Do NOT include evaluation suggestions. Do NOT include an intro sentence. 
    Output format should be exactly the example below.
    **Example**
    Overhead press

    1. Feet & base
    2. Glutes & legs
    3. Core & Ribcage
    4. Shoulder position
    5. Bar path
    6. Head & Neck
    7. Lockout position
    8. Tempo and control
    """},


        {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{classification_image}"}}
        ]}


    ])

    classified_keywords = clean_classification_text(response)

    return {"classified_keywords": classified_keywords}


In [28]:
def vector_db_node(state: GraphState):
    user_query = state["user_query"]
    classified_keywords = state.get("classified_keywords", "")
    retrieval_query = classified_keywords

    if classified_keywords:
        results_user = vectorstore.similarity_search(user_query, k=5)
        results_video = vectorstore.similarity_search(retrieval_query, k=5)
        results = results_user + results_video

    if not classified_keywords: 
        results_user = vectorstore.similarity_search(user_query, k=5)
        results = results_user

    unique_results = []
    seen = set()

    for r in results:
        content = r.page_content
        if content not in seen:
            seen.add(content)
            unique_results.append(r)


    top_k_chunks = "\n".join([r.page_content for r in unique_results])

    return {"top_k_chunks": top_k_chunks}

In [29]:
def response_generator(state: GraphState):
    top_k_chunks = state['top_k_chunks']
    encoded_images = state.get("encoded_images", [])
    session_id = state["session_id"]
    user_query = state["user_query"]
    classified_keywords = state["classified_keywords"]

    prompt = ChatPromptTemplate.from_messages([
            ("system", """You are a world-class fitness coach. You have extensive experience in helping weight lifters achieve perfect form and maximum hypertrophy. 
    Your job is to analyze images of users lifting weights, offer them advice from your context, and to answer any questions they might have. 
    Inspect each image CLOSELY and carefully for problems or issues related to best practices in exercise form. Help the user diagnose their incorrect form. 
    
        # Be specific about what you observe.
        # Go through each of the following checkpoints {classified_keywords} and assess what you see. 
        # For each checkpoint, state what you observe in the image and whether it aligns with your context.
    ---
        # ANSWER CONTEXT
    Use ONLY the following context when answering a user: 
 
    {top_k_chunks}
    
    If the query or image isn't in context, reply, 'I don't have expert coaching content for this exercise yet. 
    Currently I can analyze: bench press, overhead press, incline bench press...'"

             
        # Only offer advice about what you can clearly see. 
        # If there is no issue with the checkpoint, tell the user why it looks good, but only on the first interaction.
        # When offering follow-up advice, discuss problems checkpoints only. But remain polite and encouraging, especially if a user mentions that they are struggling with an exercise or concept.
        # If important checkpoints are hidden due to equipment or camera angle, say so.
        # If the user has good form, let them know. Use positive reinforcement, like a coach would.
        # Don't nitpick or overcorrect. 
    ---  


    """),
        MessagesPlaceholder(variable_name="history"),
        MessagesPlaceholder(variable_name="input"),
    ])



    user_msg = HumanMessage(content=[
        {"type": "text", "text": user_query},
        *encoded_images,   # <- your list of {"type":"image_url",...}
    ])

    llm = ChatOpenAI(model='gpt-5',
                    temperature=0.5)

    output_parser = StrOutputParser()

    chain = prompt | llm | output_parser

    chain_with_history = RunnableWithMessageHistory(
        chain,
        get_session_history=get_chat_history,
        input_messages_key="input",
        history_messages_key="history"

    )

    response = chain_with_history.invoke(
        {"input": [user_msg], "top_k_chunks": top_k_chunks, "classified_keywords": classified_keywords},
        config={"configurable": {"session_id": session_id}}
    )


    return {"response": response}
    

In [30]:
def chat_memory(state: GraphState):
    user_query = state["user_query"]
    session_id = state["session_id"]
    
    prompt = ChatPromptTemplate.from_messages([
            ("system", """You are a world-class fitness coach. You have extensive experience in helping weight lifters achieve perfect form and maximum hypertrophy. 
    Your job is to analyze images of users lifting weights, offer them advice from your context, and to answer any questions they might have. 


    """),
        MessagesPlaceholder(variable_name="history"),
        MessagesPlaceholder(variable_name="input"),
    ])

    user_msg = HumanMessage(content=[
        {"type": "text", "text": user_query}
    ])

    llm = ChatOpenAI(model='gpt-4o',
                    temperature=0.5)

    output_parser = StrOutputParser()

    chain = prompt | llm | output_parser

    chain_with_history = RunnableWithMessageHistory(
        chain,
        get_session_history=get_chat_history,
        input_messages_key="input",
        history_messages_key="history"

    )

    response = chain_with_history.invoke(
        {"input": [user_msg]},
        config={"configurable": {"session_id": session_id}}
    )

    return {"response": response}
    


# Router function

In [31]:
def route_query(state: GraphState):
    # first check: is there a video?
    if state["user_video"]:
        return "video_encoder"
    
    # no video — ask the LLM which path
    route = router_chain.invoke({"query": state["user_query"]})
    route = route.lower().strip()
    
    if "vectorstore" in route:
        return "vector_db"
    else:
        return "chat_memory"

# Audio transcription function

In [32]:
# def transcribe_audio(audio_path):
#     audio_file = open(audio_path, "rb") # rb = read binary. audio files are binary data (raw bytes). this tells python gto open as binary instaed of text
#     transcription = client.audio.transcriptions.create(
#         model="whisper-1",
#         file=audio_file
#     )

#     return transcription.text
    

# user_query = transcribe_audio("/Users/chandlershortlidge/Desktop/Ironhack/fitness-form-coach/data/raw_voice_notes/bench_press_voice_note.m4a")
# print(user_query)

In [33]:
# conditional edge
workflow.add_conditional_edges(START, route_query)

# add nodes
workflow.add_node("video_encoder", video_encoder_node)
workflow.add_node("video_classification_router", video_classification_node)
workflow.add_node("vector_db", vector_db_node)
workflow.add_node("response_generator", response_generator)
workflow.add_node("chat_memory", chat_memory)



# connect them
workflow.add_edge("chat_memory", END)
workflow.add_edge("video_encoder", "video_classification_router")
workflow.add_edge("video_classification_router", "vector_db")
workflow.add_edge("vector_db", "response_generator")
workflow.add_edge("response_generator", END)

app = workflow.compile()


In [34]:
result = app.invoke({
    "session_id": 123,
    "user_query": "how's my form?",
    "user_video": "/Users/chandlershortlidge/Desktop/Ironhack/fitness-form-coach/data/raw_workout_videos/bench_press/bench_07.mov"
})

print(result["response"])

Frames processed: 15, (10s cap)
Saved to processed-images/bench_07
Video name: bench_07
Thanks for the new clips. Here’s a focused check on the bench press, calling out only the issues I can clearly see:

- Feet/base
  • Heels are off the floor and your feet are set forward/away from the bench.  
  • Fix: Pull each foot back until your shins are nearly vertical and plant the heels flat. Keep the legs close to the bench when viewed from the front and drive through the heels.

- Glutes/legs
  • Your butt pops up off the pad during the press.  
  • Fix: Keep hips down so your butt touches the bench the whole set. Squeeze the glutes and use your leg drive “along the bench through the hips into the arched back” to reinforce the arch without bridging.

- Lower back/arch
  • Big arch is present; it’s fine, but it must be supported by planted glutes and heels so the chest stays high without the hips lifting.

- Grip/wrist position
  • Hard to judge from this angle; can’t clearly see wrist stac

In [35]:

# result = app.invoke({
#     "session_id": 123,
#     "user_query": user_query,
#     "user_video": ""
# })

# print(result["response"])

In [36]:
result = app.invoke({
    "session_id": 123,
    "user_query": "thanks for your help today!",
    "user_video": ""
})

print(result["response"])

You're welcome! If you have any more questions or need further assistance, feel free to reach out. Happy lifting! 🏋️‍♂️


In [37]:
# result = app.invoke({
#     "session_id": 123,
#     "user_query": "can you tell me again how to make it easier?",
#     "user_video": ""
# })

# print(result["response"])

In [38]:
def correctness_evaluator(run, example):
    generated = run.outputs["answer"]
    reference = example.outputs["answer"]
    question = example.inputs["user_query"]

    prompt = f"""You are evaluating a fitness coaching AI assistant.
Your job is to assess the quality of its response against a reference answer.

Question: {question}
Reference answer: {reference}
Generated answer: {generated}

Scoring criteria:

Score 0 — The generated answer contains incorrect or potentially harmful advice that contradicts the reference answer. This is the worst outcome.
Score 1 — The generated answer is incomplete or omits important details from the reference, but contains nothing incorrect or harmful.
Score 2 — The generated answer is correct and captures the key facts from the reference answer.

Important: Penalize incorrect advice more harshly than omission. A response that says nothing is better than one that says something wrong.

Reply with only the number: 0, 1, or 2."""

    from openai import OpenAI
    openai_client = OpenAI()
    result = openai_client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}]
    )
    try: 
        score_raw = result.choices[0].message.content.strip()
        score = int(''.join(filter(str.isdigit, score_raw))[0])
    except:
        score = -1
    return {"key": "correctness", "score": score / 2} # normalize the score so it's <= 1

In [39]:
def run_rag_pipeline(inputs: dict, attachments: dict) -> dict:
    # grab video from attachments in LangSmith
    video_name = list(attachments.keys())[0] 
    # read the video as raw bytes
    video_bytes = attachments[video_name]["reader"].read()
    
    # save to temp file because analyze_video expects a filepath
    with tempfile.NamedTemporaryFile(suffix=".mp4", delete=False) as tmp:
        # . delete=False means "don't delete this file when we're done writing" because we still need it for the pipeline.
        tmp.write(video_bytes)
        tmp_path = tmp.name
    
    state = {
        "user_query": inputs["user_query"],
        "session_id": "eval-session",
        "classified_keywords": "",
        "encoded_images": [],
        "user_video": tmp_path
    }
    
    state = {**state, **video_encoder_node(state)}
    state = {**state, **video_classification_node(state)}
    state = {**state, **vector_db_node(state)}
    state = {**state, **response_generator(state)}

    print("STATE KEYS:", state.keys())
    print("RESPONSE:", state.get("response", "NOT FOUND"))
    
    return {"answer": state["response"]}

In [ ]:
from langsmith import Client
from langsmith.evaluation import evaluate

langsmith_client = Client()

os.environ["LANGCHAIN_TRACING_V2"] = "false" 
# stops LangSmith from trying to log all the big intermediate data
# evaluate() function will still capture your inputs, outputs, and evaluator scores
# it just won't try to upload the full trace with all those heavy frames.

results = evaluate(
    run_rag_pipeline,
    data="Image assessment response accuracy v1",
    evaluators=[correctness_evaluator],
    experiment_prefix="vision-faithfullness-rag-v1"
)

View the evaluation results for experiment: 'vision-faithfullness-rag-v1-a25bc844' at:
https://smith.langchain.com/o/5dffb1fc-82e5-4000-a9a5-d9c923e0f73c/datasets/1be4093e-6f7d-4021-a94b-d6cf76c22b6a/compare?selectedSessions=56621e94-87a6-473f-bfc1-ec698aa4b9bb




0it [00:00, ?it/s]

Frames processed: 15, (10s cap)
Saved to processed-images/tmppzshuec1
Video name: tmppzshuec1


Failed to send compressed multipart ingest: Connection error caused failure to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. The content length of 222639812 bytes exceeds the maximum size limit of 20971520 bytes. SSLError(MaxRetryError("HTTPSConnectionPool(host='api.smith.langchain.com', port=443): Max retries exceeded with url: /runs/multipart (Caused by SSLError(SSLError(5, '[SYS] unknown error (_ssl.c:2426)')))"))
Content-Length: 222639812
API Key: lsv2_********************************************cctrace=019cdd57-ba19-71a3-a18f-ad9f0341ec20,id=019cdd57-cbc0-7492-b9b5-90f832552a51; trace=019cdd57-ba19-71a3-a18f-ad9f0341ec20,id=019cdd57-cbc5-7682-9061-05ddfb85fbc8; trace=019cdd57-ba19-71a3-a18f-ad9f0341ec20,id=019cdd57-cbcc-7de3-abf0-a1c9f549cfa3; trace=019cdd57-ba19-71a3-a18f-ad9f0341ec20,id=019cdd57-cbd1-7500-801d-e64f215de4eb; trace=019cdd57-ba19-71a3-a18f-ad9f0341ec20,id=019cdd57-cbd1-7500-801d-e64f215de4eb; trace=019cdd57-ba19-71a3-a18f-ad9f0341ec20,id=019

STATE KEYS: dict_keys(['user_query', 'session_id', 'classified_keywords', 'encoded_images', 'user_video', 'classification_image', 'top_k_chunks', 'response'])
RESPONSE: Thanks for the front‑angle clips—solid work. You’re moving the bar with control, touching the chest each rep, and keeping your hips down.

Checkpoint-by-checkpoint

- Feet/base: Your feet aren’t visible from this angle, so I can’t assess. Make sure they’re set back as far as you comfortably can while staying flat and close to the bench when viewed from the front. This helps you use leg drive.

- Glutes/legs: Glutes look planted—good. On the press, squeeze your glutes and drive through the heels to reinforce the arch.

- Arch/back position: You’ve got a nice arch and high chest—good. Keep it by staying tight through the whole set.

- Shoulder position: Shoulders look fairly stable. Keep the shoulder blades pinched together and use the upper back as the platform to press from.

- Grip/wrist: Wrists appear bent back at loc

...

Error running target function: [Errno 2] No such file or directory: '/Users/chandlershortlidge/Desktop/Ironhack/fitness-form-coach/data/processed/processed-images/tmpjnwga6tv_14.jpg'
Traceback (most recent call last):
  File "/opt/miniconda3/envs/fitness-form-coach/lib/python3.10/site-packages/langsmith/evaluation/_runner.py", line 1903, in _forward
    fn(*args, langsmith_extra=langsmith_extra)
  File "/opt/miniconda3/envs/fitness-form-coach/lib/python3.10/site-packages/langsmith/run_helpers.py", line 758, in wrapper
    function_result = run_container["context"].run(
  File "/var/folders/2n/c95ntf5d1wx1_vrd1b3ljbkw0000gn/T/ipykernel_48184/719908470.py", line 21, in run_rag_pipeline
    state = {**state, **video_encoder_node(state)}
  File "/var/folders/2n/c95ntf5d1wx1_vrd1b3ljbkw0000gn/T/ipykernel_48184/436431845.py", line 3, in video_encoder_node
    user_video_encoded = analyze_video(filepath_in=user_video, frame_count=15, max_seconds=10) # encodes video
  File "/Users/chandlershor

Frames processed: 15, (10s cap)
Saved to processed-images/tmpjnwga6tv
Video name: tmpjnwga6tv
Frames processed: 15, (10s cap)
Saved to processed-images/tmp7s4d0ez8
Video name: tmp7s4d0ez8


Failed to send compressed multipart ingest: Connection error caused failure to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. The content length of 40797079 bytes exceeds the maximum size limit of 20971520 bytes. SSLError(MaxRetryError("HTTPSConnectionPool(host='api.smith.langchain.com', port=443): Max retries exceeded with url: /runs/multipart (Caused by SSLError(SSLError(5, '[SYS] unknown error (_ssl.c:2426)')))"))
Content-Length: 40797079
API Key: lsv2_********************************************cctrace=019cdd6a-a467-70a1-8b4e-d1596879dc80,id=019cdd6a-bd73-7810-98fc-a998b08ce2c4; trace=019cdd6a-a467-70a1-8b4e-d1596879dc80,id=019cdd6a-bced-7661-95a6-0b4e49a68af8
Failed to send compressed multipart ingest: Connection error caused failure to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. The content length of 37800561 bytes exceeds the maximum size limit of 20971520 bytes. SSLError(MaxRetryError("HTTPSConnectionPool(host='api.smith.langchai

STATE KEYS: dict_keys(['user_query', 'session_id', 'classified_keywords', 'encoded_images', 'user_video', 'classification_image', 'top_k_chunks', 'response'])
RESPONSE: Thanks for the angle—good work keeping the bar under control and your butt on the bench. Here’s your checkpoint-by-checkpoint review.

Bench Press checkpoints

- Feet/base
  - What I see: Feet set wide and forward with little visible leg drive.
  - Aligns? Not ideal. Set your feet back as far as you comfortably can while keeping them flat and close to the bench when viewed from the front. Keep steady pressure into the floor.

- Glutes/legs
  - What I see: Glutes stay down (good), but legs look relaxed.
  - Aligns? Partly. Squeeze your glutes and keep light knees‑out tension to create a solid base.

- Back arch
  - What I see: Mild/flat arch.
  - Aligns? Could be better. Puff the chest up and maintain the arch throughout the set.

- Shoulder blades
  - What I see: Upper back looks a bit loose at the bottom.
  - Aligns? N

Failed to send compressed multipart ingest: Connection error caused failure to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. The content length of 79628763 bytes exceeds the maximum size limit of 20971520 bytes. SSLError(MaxRetryError("HTTPSConnectionPool(host='api.smith.langchain.com', port=443): Max retries exceeded with url: /runs/multipart (Caused by SSLError(SSLError(5, '[SYS] unknown error (_ssl.c:2426)')))"))
Content-Length: 79628763
API Key: lsv2_********************************************cctrace=019cdd6e-30d7-7591-bf07-b7246e54a6e2,id=019cdd71-47ad-7480-b193-7dee91ecb758; trace=019cdd6e-30d7-7591-bf07-b7246e54a6e2,id=019cdd6e-4560-79d3-b3fa-0a8b9f9b62c9; trace=019cdd6e-30d7-7591-bf07-b7246e54a6e2,id=019cdd6e-4542-7342-a276-a3ac18c503a0
Failed to send compressed multipart ingest: Connection error caused failure to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. The content length of 39815706 bytes exceeds the maximum size limit of 